In [1]:
from scipy.optimize import minimize, rosen, rosen_der,approx_fprime

In [2]:
from numpy.random import rand

In [3]:
import numpy as np

In [4]:
def objective_functionVec(X, H, G, A, lamda):
    # Regularization
    term1 = np.sum(np.power(X - np.ones(X.shape), 2))
    # Occlusion based penalty

    mat_C = X * np.power(np.ones(N)-G,lamda) * np.ones((N,N))
    mat_D = G * np.ones((N,N))
    term2 = np.sum(np.power(mat_C * H.transpose() * mat_D , 2))
    
    # Occlusion based penalty 2

    mat_A = X * np.power(np.ones(N)-G,lamda) * np.ones((N,N))
    mat_B = G * np.ones((N,N))
    term3 = np.sum(np.power(mat_A * H * mat_B , 2))

    # smoothness

    mat_E = X * np.ones((N,N))
    mat_F = X.transpose() * np.ones((N,N))
    term4 = np.sum(A * np.power(mat_E - mat_F, 2))
    
    return term1 + term2 + term3 + term4

In [11]:
from numpy import array, dot
from qpsolvers import solve_qp

linesNo = 1000
sigmentsNo = 4
N = linesNo * sigmentsNo

# G is the local importance => 0.5
# A adjacency matrix
# H occlusion matrix

np.random.seed(10)
O = np.random.random(N)

G = 0.5 * np.ones(N)
G = np.diag(G)
H = np.random.random((N,N))
A = np.random.binomial(n=1, p=0.5, size=[N,N])
I = np.identity(N)
D = A # change this to the backward difference operator

#{ 1.0f, 2.0f, 0.2f, 0.3f, 5.0f };//p, q, r, s, lambda
coff1_p = 1.0
coff2_q = 2.0
coff3_r = 0.2
coff4_s = 0.3
coff5_lamda = 5.0

W = np.power((I-G),coff5_lamda) * H * G

#print((I-G))
#print(H)
print(G)
#print(H*G)
#print(W)

Q = coff1_p * I + coff2_q * (W * W.T) + coff3_r * (W.T * W) ;#+ coff4_s * (D.T * D);

c = -1 * np.ones(N);
lb = np.zeros(N);
ub = np.ones(N);

print(Q)
print(Q.shape, c.shape, lb.shape,ub.shape)

x = solve_qp(Q, c, lb=lb,ub=ub, solver="quadprog")
print(x.shape)
print(f"QP solution: x = {x}")


[[0.5 0.  0.  ... 0.  0.  0. ]
 [0.  0.5 0.  ... 0.  0.  0. ]
 [0.  0.  0.5 ... 0.  0.  0. ]
 ...
 [0.  0.  0.  ... 0.5 0.  0. ]
 [0.  0.  0.  ... 0.  0.5 0. ]
 [0.  0.  0.  ... 0.  0.  0.5]]
[[1.0002102  0.         0.         ... 0.         0.         0.        ]
 [0.         1.00001477 0.         ... 0.         0.         0.        ]
 [0.         0.         1.00020735 ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 1.00044935 0.         0.        ]
 [0.         0.         0.         ... 0.         1.00000866 0.        ]
 [0.         0.         0.         ... 0.         0.         1.00045586]]
(4000, 4000) (4000,) (4000,) (4000,)
(4000,)
QP solution: x = [0.99978984 0.99998523 0.99979269 ... 0.99955085 0.99999134 0.99954435]


In [18]:
np.reshape(x, (int(N/sigmentsNo),sigmentsNo))

array([[0.99978984, 0.99998523, 0.99979269, 0.9996295 ],
       [0.99981759, 0.99994196, 0.99957471, 0.9995682 ],
       [0.99998222, 0.99999427, 0.99999856, 0.9995285 ],
       ...,
       [0.99990701, 0.99987536, 0.99999844, 0.99999172],
       [0.99950487, 0.9997693 , 0.99999989, 0.99999408],
       [0.99993837, 0.99955085, 0.99999134, 0.99954435]])

In [17]:
int(N/sigmentsNo)

1000